# BaseAgent - CardioKB

Chronical records of BaseAgent's attempt in building CardioKB — a cardiovascular disease knowledge graph derived from the AlzKB template.

In [ ]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import os

from BaseAgent import BaseAgent, AgentTeam, MaxRoundsExceededError
from BaseAgent.agent_spec import AgentSpec



BASEAGENT_DIR = os.path.expanduser('~/Desktop/BaseAgent')
CARDIO_KB_DIR = os.path.expanduser('~/Desktop/Cardio-KB')
CARDIO_KB_RDF = os.path.expanduser('~/Desktop/Cardio-KB/ontology/cardiokb_ontology.rdf')
CARDIO_KB_NODE_FILE = os.path.expanduser('~/Desktop/Cardio-KB/ontology/schema/node_types.txt')
CARDIO_KB_EDGE_FILE = os.path.expanduser('~/Desktop/Cardio-KB/ontology/schema/edge_types.txt')

MCP_CONFIG = f"{BASEAGENT_DIR}/examples/mcp_config.yaml"
SKILLS_DIR = f"{BASEAGENT_DIR}/skills"

THREAD_ID = "cardiokb"

CVD_DISEASE_LIST_FILE = os.path.expanduser('~/Desktop/Cardio-KB/ontology/diseases/cvd.txt')

In [ ]:
def _print_token_summary(agents: list):
    """Print per-agent and total token counts from accumulated usage_metrics."""
    print("\n=== Token usage ===")
    totals = {"input": 0, "output": 0, "total": 0}
    for agent in agents:
        metrics = agent.usage_metrics
        input_tokens = sum(m.input_tokens or 0 for m in metrics)
        output_tokens = sum(m.output_tokens or 0 for m in metrics)
        total_tokens = sum(m.total_tokens or 0 for m in metrics)
        cost = sum(m.cost or 0.0 for m in metrics)
        cost_str = f"  ${cost:.4f}" if cost else ""
        print(f"  {agent.spec.name}: {input_tokens} in / {output_tokens} out / {total_tokens} total{cost_str}")
        totals["input"] += input_tokens
        totals["output"] += output_tokens
        totals["total"] += total_tokens
    print(f"  {'─' * 40}")
    print(f"  all agents:  {totals['input']} in / {totals['output']} out / {totals['total']} total")

# Build specialist AI agents

Each unit (init, pharmgkb, clinicaltrials) uses its own `thread_id` so the 50-round budget is independent per unit. Set `BASE_AGENT_CHECKPOINT_DB=cardiokb.db` to persist conversation history across sessions.

In [ ]:
# ----- Ontology Agent -----
ontology_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="ontology_agent",
        role=(
            f"A biomedical ontology engineer managing the OWL schema and project configuration. "
            f"You own {CARDIO_KB_DIR}/data/ontology/ontology.rdf and {CARDIO_KB_DIR}/config/project.yaml: update disease_scope "
            f"(primary_terms, umls_cuis, doid_ids, mesh_ids) and keep node_types/edge_types in "
            f"sync with the RDF. Only modify the RDF on explicit request. Never edit Python source files."
            "Never read the full RDF into memory - use rdflib to query only the relevant sections for each task."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["ontology-protocol"],
    ),
    require_approval="never",
)
ontology_agent.add_mcp(MCP_CONFIG)

# ----- Database Agent -----
database_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="database_agent",
        role=(
            f"A bioinformatics data engineer managing {CARDIO_KB_DIR}/config/databases.yaml. "
            f"You evaluate biomedical data sources and set enabled flags for the target disease. "
            f"Never edit parsers or ontology mappings."
            "Never read the full RDF, TSV, or database dumps into memory - use streaming or chunking to process only the relevant sections for each task."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["database-protocol"],
    ),
    require_approval="never",
)
database_agent.add_mcp(MCP_CONFIG)

# ----- Engineer Agent -----
engineer_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="engineer_agent",
        role=(
            f"A Python software engineer writing parsers under {CARDIO_KB_DIR}/src/parsers/. "
            f"Each parser inherits from BaseParser and downloads data from one biomedical source, "
            f"returning clean pandas DataFrames. Follow the full registration checklist: "
            f"{CARDIO_KB_DIR}/src/parsers/__init__.py and {CARDIO_KB_DIR}/src/main.py PARSERS dict. "
            f"Run `python {CARDIO_KB_DIR}/src/main.py --source <name>` to verify each parser produces TSVs. "
            f"Never hardcode any disease-specific values in the parser code. "
            f"Never hardcode any credentials."
            f"Never modify OWL files or ontology_mappings.yaml."
            "Never read the full RDF, TSV, or database dumps into memory - use streaming or chunking to process only the relevant sections for each task."
        ),
        llm="azure-claude-sonnet-4-6",
        skill_names=["parser-protocol"],
    ),
    require_approval="never",
)
engineer_agent.add_mcp(MCP_CONFIG)

# ----- Mapping Agent -----
mapping_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="mapping_agent",
        role=(
            f"A knowledge graph mapping specialist owning {CARDIO_KB_DIR}/config/ontology_mappings.yaml. "
            f"You map processed TSV columns to OWL node types and relationship types. "
            f"Other useful information such as gene descriptions, annotations, structures, and cross-references is included as properties. "
            f"Always place node entries before relationship entries. "
            f"Verify all OWL names against {CARDIO_KB_DIR}/config/project.yaml node_types/edge_types before writing. "
            f"Never edit Python source files."
            "Never read the full RDF, TSV, or database dumps into memory - use streaming or chunking to process only the relevant sections for each task."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["mapping-protocol"],
    ),
    require_approval="never",
)
mapping_agent.add_mcp(MCP_CONFIG)

# ----- Memgraph Agent -----
memgraph_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="memgraph_agent",
        role=(
            f"A graph database engineer who runs the full pipeline and validates graph export. "
            f"Run `python {CARDIO_KB_DIR}/src/main.py` inside the repo to produce {CARDIO_KB_DIR}/data/output/ontology_populated.rdf."
            "Never read the full RDF into memory - use rdflib to query only the relevant sections for each task. "
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["memgraph-protocol"],
    ),
    require_approval="never",
)
memgraph_agent.add_mcp(MCP_CONFIG)

# ----- Evaluator Agent -----
evaluator_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="evaluator_agent",
        role=(
            f"A KG quality evaluator who runs {CARDIO_KB_DIR}/eval/eval_after_parser.py, {CARDIO_KB_DIR}/eval/eval_after_ontology.py, "
            f"and {CARDIO_KB_DIR}/eval/eval_after_memgraph.py in sequence. Report tier-1 blocking failures "
            f"(zero node/edge counts, OWL conformance < 1.0) and overall KG quality. "
            f"Flag any blocking failures that must be resolved before the KG is used."
            "Never read the full RDF, TSV, or database dumps into memory - use streaming or chunking to process only the relevant sections for each task. "
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["evaluation-protocol"],
    ),
    require_approval="never",
)
evaluator_agent.add_mcp(MCP_CONFIG)

# ----- HITL Agent -----
hitl_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="hitl_agent",
        role=(
            "A human review coordinator. "
            "Call `ask_user` with a drafted message that summarizes the previous agent's output in less than 5 bullet."
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["hitl-protocol"],
    ),
    require_approval="never",
)
hitl_agent.add_mcp(MCP_CONFIG)

# ----- Graph Exporter Agent -----
graph_exporter_agent = BaseAgent(
    skills_directory=SKILLS_DIR,
    spec=AgentSpec(
        name="graph_exporter_agent",
        role=(
            f"A graph database engineer who runs the full pipeline and validates graph export. "
            f"Run `python {CARDIO_KB_DIR}/src/main.py` inside the repo to produce {CARDIO_KB_DIR}/data/output/ontology_populated.rdf, "
            f"then run MemgraphExporter. Validate import.cypher (INDEX statements, LOAD CSV paths, "
            f"globally unique node IDs). Provide docker run import instructions."
            "Never read the full RDF, TSV, or database dumps into memory - use streaming or chunking to process only the relevant sections for each task. "
        ),
        llm="azure-claude-haiku-4-5",
        skill_names=["memgraph-protocol"],
    ),
    require_approval="never",
)
graph_exporter_agent.add_mcp(MCP_CONFIG)

# Task Description

In [ ]:


task=f"""
Coordinate a team of agents to BUILD the CardioKB knowledge graph at {CARDIO_KB_DIR} based on the template in {BASEAGENT_DIR}/template/.

================================================================================
AGENT RESPONSIBILITIES
================================================================================

- ontology_agent: Manage OWL schema in data/ontology/cardiokb.rdf and config/project.yaml — ensure the project strictly conforms to the ontology.
- database_agent: Manage config/databases.yaml — add the pharmgkb data source entry.
- engineer_agent: Manage parsers in src/parsers/.
- mapping_agent: Manage config/ontology_mappings.yaml — ensure correct configurations (RDF classes, data property map, and merge/cross-reference).
- evaluator_agent: Focus on parser-specific evaluation by running only eval_after_parser script and report the quality summary. List any tier-1 blocking failures explicitly.
- graph_exporter_agent: Run the full pipeline with `python src/main.py` to produce data/output/ontology_populated.rdf. Validate import.cypher and provide docker run import instructions.

================================================================================
PROJECT INITIALIZATION
================================================================================
1. Copy the template directory from {BASEAGENT_DIR}/template/ to {CARDIO_KB_DIR}/.
2. Update {CARDIO_KB_DIR}/config/project.yaml with CardioKB-specific disease scope specified in {CVD_DISEASE_LIST_FILE}.
3. Update {CARDIO_KB_DIR}/data/ontology/ontology.rdf to ensure the existence all necessary classes mentioned in {CARDIO_KB_NODE_FILE} and {CARDIO_KB_EDGE_FILE}.

================================================================================
INSTRUCTIONS
================================================================================

CardioKB integrates 24 biomedical data sources. Instructions differ between sources with and without templates.

**Sources to ADAPT from templates:**
Adapt the template parsers in {CARDIO_KB_DIR}/src/parsers/ for CardioKB's specific scope.
1. NCBI Gene
2. DoRothEA
3. DrugBank
4. Disease Ontology
5. Gene Ontology
6. Uberon
7. MeSH
8. DrugCentral
9. BindingDB
10. CTD (Comparative Toxicogenomics Database)
11. Bgee
12. STRING
13. Reactome
14. MedLine

Instructions for sources WITH templates:
1. Use template from {CARDIO_KB_DIR}/src/parsers/<template_file>


**Sources to write FROM SCRATCH (no templates exist):**
1. ClinicalTrials.gov
2. ClinPGx
3. SIDER
4. LINCS L1000
5. Jensen TISSUES
6. PubTator Central
7. OpenTargets
8. HPO (Human Phenotype Ontology)
9. HGNC Gene Families
10. ClinVar

Instructions for sources WITHOUT templates:
1. Call the right agents to update configs/*.yaml files
2. The engineer agent should creates new parsers
3. Ensure the sources can be merged to other sources in the graph through cross-references or shared identifiers.

================================================================================
THE 24 DATA SOURCES
================================================================================

1. ClinicalTrials.gov
    URL: https://clinicaltrials.gov/api/v2/studies
    Access: REST API v2
    Provides: Clinical trials (ClinicalTrial nodes), conditions studied, interventions tested
    Template: NONE - write from scratch
2. ClinPGx (PharmGKB successor)
    URL: https://api.cpicpgx.org/v1
    Access: REST API
    Provides: Pharmacogenomics: gene-drug interactions, CPIC guidelines, drug labels
    Template: NONE - write from scratch
3. SIDER (Side Effect Resource)
    URL: http://sideeffects.embl.de/media/download/meddra_all_se.tsv.gz
    Access: TSV download (legacy - may need Hetionet preprocessed files)
    Provides: Drug side effects (SideEffect nodes)
    Template: hetionet_legacy
4. LINCS L1000
    URL: https://maayanlab.cloud/sigcom-lincs/ (legacy)
    Access: Preprocessed files (legacy)
    Provides: Drug-induced gene expression changes
    Template: hetionet_legacy
5. Jensen TISSUES
    URL: https://download.jensenlab.org/human_tissue_experiments_full.tsv
    Access: TSV download
    Provides: Gene-tissue associations
    Template: tissues_parser.py
6. PubTator Central
    URL: https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTatorCentral/gene2pubtatorcentral.gz
    Access: FTP download
    Provides: Literature-mined gene-disease associations
    Template: pubtator_parser.py
7. OpenTargets
    URL: https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/latest/output/etl/parquet/
    Access: Parquet download
    Provides: Gene-disease associations with evidence scores
    Template: opentargets_parser.py
8. HPO (Human Phenotype Ontology)
    URL: https://hpo.jax.org/data/annotations | http://purl.obolibrary.org/obo/hp.obo
    Access: OBO + annotation files
    Provides: Phenotype nodes, gene-phenotype associations
    Template: hpo_parser.py
9. HGNC Gene Families
    URL: https://www.genenames.org/cgi-bin/genegroup/download-all
    Access: TSV download
    Provides: Gene family nodes, gene-family memberships
    Template: hgnc_parser.py
10. ClinVar
    URL: https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz
    Access: FTP download
    Provides: Variant nodes with clinical significance
    Template: clinvar_parser.py

================================================================================
CONSTRAINTS
================================================================================

- Do not include any filter (e.g., disease, tissue, or drug) if the specific parser doesn't include these RDF classes.
- Directly retrieve data from the source database. Do not use any precomputed third-party processed data. Do not use any precomputed data from Hetionet.
- config/project.yaml sets project.name, display_name, and all disease_scope fields (primary_terms, umls_cuis, doid_ids, mesh_ids). 
- The databases.yaml key, PARSERS key, PARSER_CLASS_MAP key, ontology_mappings.yaml prefix, and data/processed/ subdirectory name must all be identical strings.
- In ontology_mappings.yaml, all node entries must precede all relationship entries.
- OWL names in ontology_mappings.yaml must be active (uncommented) in project.yaml.
- Credentials use the _env suffix in databases.yaml args; the parser constructor must accept the stripped parameter name.
- Do minimal changes.

"""

# Build the whole CardioKB

In [ ]:
agents = [
    ontology_agent, database_agent, engineer_agent, evaluator_agent, mapping_agent, graph_exporter_agent,
]

team = AgentTeam(
    agents=agents,
    supervisor_llm="azure-claude-opus-4-6",
    max_rounds=100,
)

In [ ]:
_log, result = team.run_sync(task, thread_id=THREAD_ID)

In [ ]:
# In case of interruption or MaxRoundsExceededError, you can continue the conversation with:
# team = AgentTeam(
#     agents=agents,
#     supervisor_llm="azure-claude-opus-4-6",
#     max_rounds=100,
# )
# _log, result = team.run_sync(f"You are previously working on the following task before reaching a max rounds limit: {task}. Identify the latest progress and continue.", thread_id=THREAD_ID)

# Sub-agent follow-up

In [ ]:
team_tid = team._current_thread_id
_log2, result2 = evaluator_agent.run(
    f"Report the evaluation status of the {TARGET_DATABASE} parser",
    thread_id=f"{team_tid}:evaluator_agent"
)

# Token usage summary

In [ ]:
_print_token_summary(agents)